# Estudo de caso — Redes neurais aplicadas a um motor elétrico PMSM

**Disciplina:** Inteligência Artificial aplicada à Engenharia Elétrica · **Projeto 1 (extensão com dados reais)**

**Problema:** compreender regimes de operação de um motor elétrico e estimar a temperatura do ímã permanente do rotor (coluna `pm`) a partir de variáveis operacionais. Dois algoritmos são construídos com NumPy: uma **SOM não supervisionada** e uma **MLP supervisionada** com retropropagação. Não usamos bibliotecas de treinamento de redes.

**Fonte real:** [Electric Motor Temperature — Kaggle](https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature), medições de bancada de um PMSM, 185 h a 2 Hz, sessões identificadas por `profile_id`, arquivo `measures_v2.csv`. Autor do conjunto: Kirchgässner / LEA Paderborn. Licença informada pelo Kaggle: CC BY-SA 4.0. **O CSV não é redistribuído neste repositório.**

**Importante:** não existem resultados pré-calculados aqui. Caso o CSV esteja ausente, o notebook explica o acesso e **não inventa curvas, temperaturas ou métricas**. Antes da aula baixe o CSV. A parte anterior do Projeto 1 (perceptron e SOM 5×7) funciona sem esse arquivo.

**Objetivos de aprendizagem:** (1) interpretar grandezas físicas e sensores; (2) separar treino/validação/teste por `profile_id`; (3) padronizar com dados de treino; (4) implementar SOM; (5) implementar MLP tanh 7→16→1 e derivadas; (6) comparar erros MAE/RMSE e discutir memória térmica. **Não usar o exemplo como dispositivo de proteção ou classificação de falhas.**

## Antes de começar

### Objetivo
Aplicar a SOM (regimes de operação) e a MLP (estimação da temperatura do ímã) a dados **reais** de um motor PMSM e avaliar os erros por sessão.

### Contexto do problema
O conjunto *Electric Motor Temperature* traz medições de um motor síncrono de ímãs permanentes. A temperatura do ímã (`pm`) é o alvo a estimar; `profile_id` identifica a sessão e não é atributo.

### Pré-requisitos
Ter feito as aulas 1 e 2 (perceptron, MLP e SOM) e ter o arquivo `measures_v2.csv`. **Sem o arquivo o notebook não produz resultados**: ele não usa dados inventados.

### Como executar
- **Kaggle (recomendado):** crie um notebook em kaggle.com/code, importe este arquivo (*File → Import Notebook*), clique em *Add Input* e adicione o conjunto https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature. Confira o caminho `/kaggle/input/…/measures_v2.csv` e execute as células na ordem; salve com *Save Version*.
- **Colab/Jupyter:** baixe `measures_v2.csv` no Kaggle e envie para o ambiente (no Colab, painel de pastas) ou defina `MOTOR_DATA_CSV`.
- Passo a passo: https://furriel.github.io/IA/apoio/executar-python.html

### Dados e parâmetros
Dados: `measures_v2.csv` (centenas de MB). Parâmetros: `SAMPLE_STRIDE` (amostragem sistemática; valores menores usam mais dados e demoram mais) e as configurações da SOM e da MLP nas seções 4 e 5.

## 1. Preparação e disponibilidade do arquivo

No Kaggle, anexe o dataset **Electric Motor Temperature** via *Add Input*. Para Colab ou Jupyter, baixe `measures_v2.csv` e faça upload/montagem. O arquivo tem centenas de MB; pode levar algum tempo para carregar. Evitamos leitura integral para memória com `chunksize` e amostragem sistemática. Modifique `SAMPLE_STRIDE` para maior fidelidade, considerando o custo computacional.

A coluna `pm` é a grandeza a **estimar**, não uma entrada da MLP. `profile_id` identifica uma sessão, **não é atributo preditivo**. O alvo `pm` pode depender da história térmica; esse modelo instantâneo é simplificado.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261105
SAMPLE_STRIDE = 20  # usa 1 a cada 20 linhas; NÃO inventa valores
MAX_TRAIN, MAX_VAL, MAX_TEST = 8000, 2000, 2000
FEATURES = ["ambient", "coolant", "u_d", "u_q", "motor_speed", "i_d", "i_q"]
TARGET = "pm"  # temperatura do ímã permanente / rotor
SOM_FEATURES = ["motor_speed", "i_mag", "coolant"]
NEEDED = ["profile_id"] + FEATURES + [TARGET]
rng = np.random.default_rng(SEED)
print("Configuração reprodutível:", {"seed": SEED, "amostragem": SAMPLE_STRIDE,
  "MLP_entradas": FEATURES, "alvo_C": TARGET})

In [ ]:
# É necessário fornecer o CSV real. Nunca substituímos por dados inventados.
# Kaggle: Add Input → wkirgsn/electric-motor-temperature.
# Colab: faça upload do measures_v2.csv ou monte o Google Drive.
# Jupyter: coloque measures_v2.csv ao lado do notebook.
manual = os.environ.get("MOTOR_DATA_CSV", "").strip()
roots = [Path("/kaggle/input"), Path.cwd(), Path("/content")]
found = []
if manual and Path(manual).expanduser().is_file():
    found.append(Path(manual).expanduser())
for root in roots:
    if root.is_dir():
        direct = root / "measures_v2.csv"
        if direct.is_file(): found.append(direct)
        if root == Path("/kaggle/input"):
            found.extend(root.glob("*/measures_v2.csv"))
            found.extend(root.glob("*/*/measures_v2.csv"))
csv_path = next((p for p in found if p.is_file()), None)
DATA_OK = csv_path is not None
if DATA_OK:
    print("Dados reais localizados:", csv_path)
else:
    print("DADOS REAIS AUSENTES — análises de treinamento não serão executadas.")
    print("Faça download em https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature")
    print("Depois carregue measures_v2.csv no ambiente e execute novamente a partir desta célula.")

## 2. Inspeção física dos dados e unidades

Segundo a descrição do conjunto (versão 2, valores **não normalizados**), as grandezas usadas são:

| Coluna | Grandeza | Unidade | Papel |
|---|---|---|---|
| `u_d`, `u_q` | tensão nos eixos d e q | V | entrada da MLP |
| `i_d`, `i_q` | corrente nos eixos d e q | A | entrada da MLP; `i_mag` na SOM |
| `motor_speed` | velocidade do rotor | rpm | entrada da MLP e da SOM |
| `ambient`, `coolant` | temperaturas ambiente e do líquido de arrefecimento | °C | entradas |
| `pm` | temperatura do ímã permanente (rotor) | °C | **alvo** da MLP, nunca entrada |
| `profile_id` | identificador da sessão de medição | — | só para separar os dados |

`i_mag = √(i_d² + i_q²)` é a magnitude da corrente no plano dq, **não** a corrente eficaz de fase sem as convenções da transformação. Confira as faixas impressas abaixo: se forem compatíveis com valores normalizados (média ≈ 0, desvio ≈ 1), você está com a versão 1 do arquivo e as unidades acima não se aplicam. Não há rótulos de falha neste conjunto.

In [ ]:
if DATA_OK:
    frames = []
    n_total = 0
    for chunk in pd.read_csv(csv_path, usecols=lambda name: name in NEEDED,
                             chunksize=120_000):
        missing = sorted(set(NEEDED) - set(chunk.columns))
        if missing:
            raise ValueError(f"Colunas não encontradas no dataset: {missing}")
        n_total += len(chunk)
        # Amostragem sistemática uniforme por bloco, sem coletar milhões de linhas na RAM.
        small = chunk.iloc[::SAMPLE_STRIDE].copy()
        frames.append(small)
    df = pd.concat(frames, ignore_index=True)
    for c in NEEDED:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    n_before = len(df)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=NEEDED)
    df["i_mag"] = np.hypot(df["i_d"], df["i_q"])
    print(f"Linhas originais: {n_total:,}; amostra: {n_before:,}; válidas: {len(df):,}")
    print(f"Perfis distintos: {df.profile_id.nunique()}")
    if df.profile_id.nunique() < 6:
        raise ValueError("Exigem-se ao menos seis sessões profile_id para uma divisão segura.")
    print(df[NEEDED].describe().round(2).to_string())
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.scatter(df["motor_speed"].iloc[::10], df["i_mag"].iloc[::10],
               s=6, alpha=0.3)
    ax.set(xlabel="Velocidade do motor (rpm, versão 2)", ylabel="|i_dq| (A, versão 2)",
           title="Regimes de operação — amostra real do Kaggle")
    plt.tight_layout(); plt.show()
else:
    print("Etapa ignorada: forneça measures_v2.csv (sem resultados fictícios).")

In [ ]:
if DATA_OK:
    UNIDADES = {"u_d": "V", "u_q": "V", "i_d": "A", "i_q": "A", "motor_speed": "rpm", "ambient": "°C", "coolant": "°C", "pm": "°C"}
    faixas = df[list(UNIDADES)].agg(["min", "median", "max"]).T
    faixas["unidade"] = [UNIDADES[c] for c in faixas.index]
    print(faixas.round(1).to_string())
    if abs(df["pm"].mean()) < 2 and 0.5 < df["pm"].std() < 2:
        print("ATENÇÃO: valores parecem normalizados (versão 1). Não interprete os resultados em °C.")
    else:
        print("Faixas compatíveis com grandezas físicas (versão 2).")
else:
    print("Inspeção não executada: dataset ausente.")

## 3. Separação correta dos perfis

Não embaralhe linhas do mesmo `profile_id` entre treino e teste: amostras consecutivas possuem alta correlação. A divisão por sessões inteiras simula melhor a avaliação em ciclos não vistos. Treino define todas as médias e desvios da normalização. Validação orienta a parada antecipada; teste permanece reservado à comparação final.

In [ ]:
if DATA_OK:
    profiles = np.array(sorted(df.profile_id.unique()))
    shuffled = rng.permutation(profiles)
    n_test = max(1, int(round(len(profiles) * .15)))
    n_val = max(1, int(round(len(profiles) * .15)))
    test_ids, val_ids = shuffled[:n_test], shuffled[n_test:n_test+n_val]
    train_ids = shuffled[n_test+n_val:]
    assert len(train_ids) and set(train_ids).isdisjoint(val_ids)
    assert set(test_ids).isdisjoint(train_ids) and set(test_ids).isdisjoint(val_ids)
    def take(ids, limit):
        part = df[df.profile_id.isin(ids)]
        return part.sample(n=min(limit, len(part)), random_state=SEED).reset_index(drop=True)
    train, val, test = take(train_ids, MAX_TRAIN), take(val_ids, MAX_VAL), take(test_ids, MAX_TEST)
    assert min(map(len, (train,val,test))) > 10
    print("Sessões por subconjunto:", len(train_ids), len(val_ids), len(test_ids))
    print("Amostras:", len(train), len(val), len(test))
    print("Treino, validação e teste possuem profile_id DISJUNTOS.")
    x_mean = train[FEATURES].mean().to_numpy(float)
    x_std = train[FEATURES].std(ddof=0).replace(0,1).to_numpy(float)
    y_mean = float(train[TARGET].mean())
    y_std = max(float(train[TARGET].std(ddof=0)), 1e-8)
    norm_x = lambda frame: ((frame[FEATURES].to_numpy(float)-x_mean)/x_std)
    Xtr, Xva, Xte = norm_x(train), norm_x(val), norm_x(test)
    ytr = (train[TARGET].to_numpy(float)-y_mean)/y_std
    yva = (val[TARGET].to_numpy(float)-y_mean)/y_std
    yte_real = test[TARGET].to_numpy(float)
else:
    print("Divisão por sessões não executada: dataset ausente.")

## 4. SOM: mapa auto-organizável dos regimes de operação

Para cada amostra `x`, calculamos `c=argminⱼ ||x-mⱼ||²`. Atualizamos `mⱼ←mⱼ+α(t)·h(c,j,t)·(x-mⱼ)` com vizinhança gaussiana `h=exp(−||rⱼ-r_c||²/(2σ(t)²))`. **Treino sem classes e sem usar `pm`.**

Os gráficos mostram ocupação de unidades, erro de quantização e **temperatura média por unidade após o treino**. Essa última é uma associação descritiva posterior, não um diagnóstico nem uma previsão de temperatura pela SOM.

In [ ]:
if DATA_OK:
    # SOM 5×5, três atributos físicos, sem usar o alvo de temperatura.
    z_mean = train[SOM_FEATURES].mean().to_numpy(float)
    z_std = train[SOM_FEATURES].std(ddof=0).replace(0,1).to_numpy(float)
    Ztr = (train[SOM_FEATURES].to_numpy(float)-z_mean)/z_std
    Zte = (test[SOM_FEATURES].to_numpy(float)-z_mean)/z_std
    r = np.array([(i,j) for i in range(5) for j in range(5)], dtype=float)
    srng = np.random.default_rng(SEED+1)
    Wsom = srng.normal(0,.4,(25,len(SOM_FEATURES)))
    som_error = []
    steps = 900
    for t in range(steps):
        x = Ztr[srng.integers(0, len(Ztr))]
        distances = np.sum((Wsom-x)**2, axis=1)
        c = int(np.argmin(distances))
        alpha = .32*np.exp(-t/400)+.04
        sigma = 1.5*np.exp(-t/360)+.32
        h = np.exp(-np.sum((r-r[c])**2,axis=1)/(2*sigma*sigma))
        Wsom += alpha*h[:,None]*(x-Wsom)
        if (t+1)%30==0:
            sub = Ztr[:500]
            som_error.append(float(np.mean(np.sqrt(np.min(
                np.sum((sub[:,None,:]-Wsom[None,:,:])**2,axis=2),axis=1)))))
    bmu_test = np.argmin(np.sum((Zte[:,None,:]-Wsom[None,:,:])**2,axis=2),axis=1)
    counts = np.bincount(bmu_test,minlength=25).reshape(5,5)
    bmu_train = np.argmin(np.sum((Ztr[:,None,:]-Wsom[None,:,:])**2,axis=2),axis=1)
    occ_tr = np.bincount(bmu_train,minlength=25)/len(bmu_train)
    occ_te = np.bincount(bmu_test,minlength=25)/len(bmu_test)
    print("Unidades usadas no teste e quase vazias no treino (<0,5 %):",
          [int(k) for k in range(25) if occ_te[k] > 0.02 and occ_tr[k] < 0.005] or "nenhuma")
    # Temperatura é usada apenas na DESCRIÇÃO posterior dos grupos, não no treino.
    temp_avg = np.array([yte_real[bmu_test==k].mean() if np.any(bmu_test==k)
                         else np.nan for k in range(25)]).reshape(5,5)
    fig, axs = plt.subplots(1,3,figsize=(14,4))
    axs[0].plot(np.arange(30,steps+1,30),som_error)
    axs[0].set(title="Erro de quantização SOM",xlabel="Atualizações",ylabel="Distância média (desvios-padrão do treino)")
    a = axs[1].imshow(counts);axs[1].set(title="Ocupação por BMU (teste)",xlabel="Coluna da grade",ylabel="Linha da grade")
    fig.colorbar(a,ax=axs[1],shrink=.8)
    b = axs[2].imshow(temp_avg);axs[2].set(title="Média de pm por BMU (teste)",xlabel="Coluna",ylabel="Linha")
    fig.colorbar(b,ax=axs[2],shrink=.8,label="°C")
    plt.tight_layout();plt.show()
    print("SOM não supervisionada. A temperatura foi usada apenas para descrever agrupamentos.")
else:
    print("SOM real não executada: dataset ausente.")

## 5. MLP: estimação supervisionada de temperatura

A MLP usa sete entradas: `ambient, coolant, u_d, u_q, motor_speed, i_d, i_q`. Arquitetura: 7→16 (`tanh`)→1 (linear).

```
h = tanh(X·W1+b1)            (camada oculta)
ŷ = h·W2+b2                  (saída estimada normalizada)
L = 0,5 * média[(ŷ−y)²]      (perda)
δ2 = (ŷ−y)/N
∂L/∂W2 = hᵀ·δ2
δ1 = (δ2·W2ᵀ) ⊙ (1−h²)
∂L/∂W1 = Xᵀ·δ1
W ← W−η∂L/∂W                (gradiente descendente)
```

A parada antecipada usa apenas a validação. Não utilize temperaturas do rotor ou do estator como entradas disfarçadas do alvo. Uma MLP instantânea pode ter erro importante porque o motor apresenta inércia térmica; essa é uma limitação científica a discutir.

In [ ]:
if DATA_OK:
    # MLP autoral: 7 entradas → 16 neurônios tanh → 1 saída linear.
    # Perda: 0,5 * média[(ŷ−y)^2]. Gradientes por regra da cadeia.
    mrng = np.random.default_rng(SEED+2)
    in_dim, hidden = Xtr.shape[1], 16
    W1 = mrng.normal(0,np.sqrt(1/in_dim),(in_dim,hidden))
    b1 = np.zeros(hidden)
    W2 = mrng.normal(0,np.sqrt(1/hidden),(hidden,1))
    b2 = np.zeros(1)
    lr, batch_size, max_epochs, patience = .018, 128, 100, 14
    best, best_epoch, best_loss, since_best = None, 0, float("inf"), 0
    history_train, history_val = [], []
    def forward(X):
        h = np.tanh(X@W1+b1)
        return h, (h@W2+b2).reshape(-1)
    for epoch in range(max_epochs):
        order = mrng.permutation(len(Xtr))
        for start in range(0,len(order),batch_size):
            ix = order[start:start+batch_size]; xb,yb = Xtr[ix], ytr[ix]
            h,pred = forward(xb)
            delta = ((pred-yb)/len(xb))[:,None]  # derivada de 0.5*MSE
            gW2 = h.T@delta
            gb2 = np.sum(delta,axis=0)
            gh = (delta@W2.T)*(1-h*h)
            gW1 = xb.T@gh
            gb1 = np.sum(gh,axis=0)
            # Gradientes antes de atualizar: regra da cadeia correta.
            W2 -= lr*np.clip(gW2,-5,5);b2 -= lr*np.clip(gb2,-5,5)
            W1 -= lr*np.clip(gW1,-5,5);b1 -= lr*np.clip(gb1,-5,5)
        _,pt=forward(Xtr);_,pv=forward(Xva)
        loss_tr=float(np.mean((pt-ytr)**2))
        loss_val=float(np.mean((pv-yva)**2))
        history_train.append(loss_tr);history_val.append(loss_val)
        if loss_val < best_loss-1e-5:
            best_loss, best_epoch, since_best=loss_val, epoch+1, 0
            best=(W1.copy(),b1.copy(),W2.copy(),b2.copy())
        else:
            since_best+=1
            if since_best>=patience:break
    W1,b1,W2,b2=best
    def mlp_predict(X):
        return (np.tanh(X@W1+b1)@W2+b2).ravel()*y_std+y_mean
    predictions=mlp_predict(Xte)
    print(f"MLP treinada por {len(history_train)} épocas; melhor validação na época {best_epoch}.")
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot(history_train,label="Treino (MSE normalizado)")
    ax.plot(history_val,label="Validação (MSE normalizado)")
    ax.axvline(best_epoch-1,ls=":",label="Melhor época")
    ax.set(xlabel="Época",ylabel="MSE normalizado",title="Aprendizado da MLP")
    ax.legend();plt.tight_layout();plt.show()
else:
    print("MLP real não executada: dataset ausente.")

## 6. Comparação e interpretação

O modelo só é útil se for comparado com referências simples. Apresentamos média do treino, temperatura do líquido de arrefecimento, regressão linear com as mesmas variáveis e MLP.

**Métricas:** MAE = média(|y−ŷ|), RMSE = √média((y−ŷ)²), em °C. Menores valores são melhores; não há garantia de que a MLP supere a regressão linear. Se a MLP for pior, discuta atributos, memória térmica, diferenças entre sessões e quantidade de dados.

In [ ]:
if DATA_OK:
    # Referências transparentes: constante média do treino, coolant e regressão linear.
    def metrics(y,p):
        e=np.asarray(y)-np.asarray(p)
        return {"MAE (°C)":float(np.mean(np.abs(e))),
                "RMSE (°C)":float(np.sqrt(np.mean(e*e)))}
    # Regressão linear resolvida por mínimos quadrados nos dados de TREINO.
    XA = np.column_stack([np.ones(len(Xtr)),Xtr])
    coef = np.linalg.lstsq(XA,ytr,rcond=None)[0]
    linear = (np.column_stack([np.ones(len(Xte)),Xte])@coef)*y_std+y_mean
    baselines = {
      "Média do treino":np.full(len(test),y_mean),
      "Temperatura do coolant":test["coolant"].to_numpy(float),
      "Regressão linear":linear,
      "MLP NumPy":predictions
    }
    report = pd.DataFrame({name:metrics(yte_real,p) for name,p in baselines.items()}).T
    print(report.round(3).to_string())
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    axes[0].scatter(yte_real,predictions,s=9,alpha=.35)
    lo=min(yte_real.min(),predictions.min());hi=max(yte_real.max(),predictions.max())
    axes[0].plot([lo,hi],[lo,hi],ls="--")
    axes[0].set(xlabel="Temperatura pm medida (°C)",ylabel="Temperatura pm estimada (°C)",
                title="MLP no teste por sessões novas")
    axes[1].hist(yte_real-predictions,bins=35)
    axes[1].set(xlabel="Erro real − estimado (°C)",ylabel="Contagem",
                title="Distribuição de erros no teste")
    plt.tight_layout();plt.show()
    print("Limitação: modelo estático, sem histórico térmico. "
          "Não é monitoramento operacional validado, nem diagnóstico de falhas.")
else:
    print("Sem métricas reais: adicione o CSV para executar o estudo.")

### 6.1 Resíduos ao longo do tempo: a inércia térmica

O gráfico de dispersão mistura instantes de várias sessões. Para enxergar a dinâmica, aplicamos a MLP e a regressão linear a **uma sessão de teste inteira**, na ordem temporal (a amostragem a cada `SAMPLE_STRIDE` linhas preserva a ordem). Erros que persistem por minutos e atrasam em relação às mudanças de carga indicam que a temperatura do ímã depende da **história** da operação — algo que um modelo instantâneo não representa.

In [ ]:
if DATA_OK:
    sess = df[df.profile_id == test_ids[0]]
    Xs = norm_x(sess)
    tempo_min = np.arange(len(sess)) * SAMPLE_STRIDE / 2 / 60      # registros a 2 Hz
    est_mlp = mlp_predict(Xs)
    est_lin = (np.column_stack([np.ones(len(Xs)), Xs]) @ coef) * y_std + y_mean
    medida = sess[TARGET].to_numpy(float)
    fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    ax[0].plot(tempo_min, medida, label="pm medida", color="k")
    ax[0].plot(tempo_min, est_mlp, label="MLP", alpha=.8)
    ax[0].plot(tempo_min, est_lin, label="regressão linear", alpha=.8)
    ax[0].set(ylabel="temperatura (°C)", title=f"Sessão de teste profile_id = {test_ids[0]} (não vista no treino)"); ax[0].legend()
    ax[1].plot(tempo_min, medida - est_mlp, label="resíduo MLP"); ax[1].plot(tempo_min, medida - est_lin, label="resíduo linear")
    ax[1].axhline(0, color="k", lw=.8); ax[1].set(ylabel="medida − estimada (°C)"); ax[1].legend()
    ax[2].plot(tempo_min, sess["motor_speed"], color="#146e83"); ax[2].set(ylabel="velocidade (rpm)", xlabel="tempo desde o início da sessão (min)")
    ax2 = ax[2].twinx(); ax2.plot(tempo_min, sess["i_mag"], color="#c2570c", alpha=.6); ax2.set_ylabel("|i_dq| (A)")
    plt.tight_layout(); plt.show()
    print(f"Nesta sessão: MAE MLP = {np.mean(np.abs(medida - est_mlp)):.2f} °C; MAE linear = {np.mean(np.abs(medida - est_lin)):.2f} °C.")
    print("Resultado de UMA sessão: não generalize; compare com a tabela de métricas do conjunto de teste.")
else:
    print("Análise temporal não executada: dataset ausente.")

## 7. Questões para os alunos e limites

Fluxo do estudo: **grandezas elétricas e operacionais → regime do motor (SOM, sem rótulos) → estimação da temperatura do ímã (MLP) → avaliação dos erros (MAE, RMSE, resíduos no tempo)**.


1. Por que `profile_id` deve organizar a divisão e não entrar na MLP como atributo?
2. Qual a diferença entre temperatura usada como **rótulo descritivo pós-SOM** e temperatura usada como **alvo supervisionado da MLP**?
3. Qual é a diferença entre os 25 vetores da SOM e os 16 neurônios ocultos da MLP?
4. Por que comparar com regressão linear e com a temperatura do coolant?
5. Por que um modelo sem atrasos/histórico pode errar em transientes térmicos?
6. Como testaria generalização em um **outro motor físico**, não apenas em outra sessão do mesmo motor?

**Resumo seguro:** não há detecção de defeitos, proteção elétrica ou inferência certificada; a aplicação é acadêmica e seus resultados dependem de executar o notebook com o CSV real.

**Referências:** [dataset e licença](https://www.kaggle.com/datasets/wkirgsn/electric-motor-temperature) · [repositório original de modelagem térmica](https://github.com/wkirgsn/deep-pmsm) · [livro Russell e Norvig catalogado na Biblioteca](https://github.com/Furriel/biblioteca/tree/main/livros/inteligencia-artificial/russell-norvig-artificial-intelligence-modern-approach-2020).

## Experimentos propostos
1. Reduza `SAMPLE_STRIDE` e compare MAE e RMSE.
2. Compare a MLP com as referências simples da seção 6 e discuta o ganho.
3. Observe os resíduos no tempo (seção 6.1) e relacione com a inércia térmica.

## Questões de interpretação
Use as questões da seção 7 deste notebook.

## Conclusão
Registre o arquivo de dados usado (caminho impresso pelo notebook), os parâmetros alterados, as métricas obtidas **na sua execução** e uma limitação. Não registre números que você não tenha obtido executando o notebook com os dados reais.